BGWO's: RF
BGWO optimizes the hyperparameters of the Random Forest classifier. The fitness function evaluates the classifier's accuracy on the test set, guiding BGWO to find the best hyperparameters.

In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier

# Load the dataset
X = pd.read_excel('../minmax.xlsx')
y = pd.read_csv('../idC_with_header.csv')['Label']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the fitness function for BGWO
# the fitness function are the hyperparameters of the Random Forest model
# n_estimators - number of trees in the forest  
# max_depth - maximum depth of the tree
def fitness_function(position):
    n_estimators = int(position[0])
    max_depth = int(position[1])
    
    # Train a Random Forest model
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate the model on the test set
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

# Initialize BGWO parameters
num_wolves = 10
num_features = 2  # n_estimators and max_depth
max_iterations = 20
alpha, beta, delta = None, None, None

# Initialize the wolves' positions randomly
wolves = np.random.randint(low=[10, 1], high=[200, 20], size=(num_wolves, num_features))

# BGWO algorithm
for iteration in range(max_iterations):
    fitness = np.array([fitness_function(wolf) for wolf in wolves])
    
    # Update alpha, beta, and delta wolves
    sorted_indices = np.argsort(fitness)[::-1]
    alpha, beta, delta = wolves[sorted_indices[:3]]
    
    # Update positions of wolves
    for i in range(num_wolves):
        for j in range(num_features):
            r1, r2, r3 = np.random.rand(3)
            A1, A2, A3 = 2 * r1 - 1, 2 * r2 - 1, 2 * r3 - 1
            C1, C2, C3 = 2 * r1, 2 * r2, 2 * r3
            
            D_alpha = abs(C1 * alpha[j] - wolves[i, j])
            D_beta = abs(C2 * beta[j] - wolves[i, j])
            D_delta = abs(C3 * delta[j] - wolves[i, j])
            
            X1 = alpha[j] - A1 * D_alpha
            X2 = beta[j] - A2 * D_beta
            X3 = delta[j] - A3 * D_delta
            
            wolves[i, j] = (X1 + X2 + X3) / 3

# Evaluate the best solution
best_wolf = alpha
best_n_estimators = int(best_wolf[0])
best_max_depth = int(best_wolf[1])

final_model = RandomForestClassifier(n_estimators=best_n_estimators, max_depth=best_max_depth, random_state=42)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

# Print performance metrics
print(f"Best Hyperparameters: n_estimators={best_n_estimators}, max_depth={best_max_depth}")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

KeyboardInterrupt: 

BGWO's: SVM
BGWO optimizes the hyperparameters of the SVM classifier. The fitness function evaluates the classifier's accuracy on the test set, guiding BGWO to find the best hyperparameters.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import SVC

# Load the dataset
X = pd.read_excel('../minmax.xlsx')
y = pd.read_csv('../idC_with_header.csv')['Label']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the fitness function for BGWO
def fitness_function(position):
    C = 10 ** position[0]  # Map to a range of 10^x for C
    gamma = 10 ** position[1]  # Map to a range of 10^x for gamma
    
    # Train an SVM model
    model = SVC(C=C, gamma=gamma, kernel='rbf', random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate the model on the test set
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

# Initialize BGWO parameters
num_wolves = 10
num_features = 2  # C and gamma
max_iterations = 20
alpha, beta, delta = None, None, None

# Initialize the wolves' positions randomly (log10 scale for C and gamma)
wolves = np.random.uniform(low=[-3, -3], high=[3, 3], size=(num_wolves, num_features))

# BGWO algorithm
for iteration in range(max_iterations):
    fitness = np.array([fitness_function(wolf) for wolf in wolves])
    
    # Update alpha, beta, and delta wolves
    sorted_indices = np.argsort(fitness)[::-1]
    alpha, beta, delta = wolves[sorted_indices[:3]]
    
    # Update positions of wolves
    for i in range(num_wolves):
        for j in range(num_features):
            r1, r2, r3 = np.random.rand(3)
            A1, A2, A3 = 2 * r1 - 1, 2 * r2 - 1, 2 * r3 - 1
            C1, C2, C3 = 2 * r1, 2 * r2, 2 * r3
            
            D_alpha = abs(C1 * alpha[j] - wolves[i, j])
            D_beta = abs(C2 * beta[j] - wolves[i, j])
            D_delta = abs(C3 * delta[j] - wolves[i, j])
            
            X1 = alpha[j] - A1 * D_alpha
            X2 = beta[j] - A2 * D_beta
            X3 = delta[j] - A3 * D_delta
            
            wolves[i, j] = (X1 + X2 + X3) / 3

# Evaluate the best solution
best_wolf = alpha
best_C = 10 ** best_wolf[0]
best_gamma = 10 ** best_wolf[1]

final_model = SVC(C=best_C, gamma=best_gamma, kernel='rbf', random_state=42)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

# Print performance metrics
print(f"Best Hyperparameters: C={best_C:.6f}, gamma={best_gamma:.6f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Best Hyperparameters: C=3.926567, gamma=0.005810
Accuracy: 97.75%

Classification Report:
              precision    recall  f1-score   support

           1       1.00      1.00      1.00         2
           2       1.00      1.00      1.00         8
           3       0.86      1.00      0.92         6
           4       1.00      0.94      0.97        18
           5       0.90      1.00      0.95         9
           6       1.00      0.83      0.91         6
           7       1.00      1.00      1.00         1
           9       1.00      1.00      1.00        13
          10       1.00      1.00      1.00         1
          11       1.00      1.00      1.00         5
          12       1.00      1.00      1.00         6
          13       1.00      1.00      1.00         7
          14       1.00      1.00      1.00         7

    accuracy                           0.98        89
   macro avg       0.98      0.98      0.98        89
weighted avg       0.98      0.98      0.98 

BGWO-Based Feature Selection for Naive Bayes Classification

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import GaussianNB

# Load the dataset
X = pd.read_excel('../max.xlsx')
y = pd.read_csv('../idC_with_header.csv')['Label']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the fitness function for BGWO
def fitness_function(position):
    # Select features based on the binary position vector
    selected_features = [i for i in range(len(position)) if position[i] > 0.5]
    if len(selected_features) == 0:  # Avoid empty feature subsets
        return 0
    
    # Train a GaussianNB model
    model = GaussianNB()
    model.fit(X_train.iloc[:, selected_features], y_train)
    
    # Evaluate the model on the test set
    y_pred = model.predict(X_test.iloc[:, selected_features])
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

# Initialize BGWO parameters
num_wolves = 10
num_features = X.shape[1]  # Number of features in the dataset
max_iterations = 20
alpha, beta, delta = None, None, None

# Initialize the wolves' positions randomly (binary positions for feature selection)
wolves = np.random.randint(0, 2, size=(num_wolves, num_features))

# BGWO algorithm
for iteration in range(max_iterations):
    fitness = np.array([fitness_function(wolf) for wolf in wolves])
    
    # Update alpha, beta, and delta wolves
    sorted_indices = np.argsort(fitness)[::-1]
    alpha, beta, delta = wolves[sorted_indices[:3]]
    
    # Update positions of wolves
    for i in range(num_wolves):
        for j in range(num_features):
            r1, r2, r3 = np.random.rand(3)
            A1, A2, A3 = 2 * r1 - 1, 2 * r2 - 1, 2 * r3 - 1
            C1, C2, C3 = 2 * r1, 2 * r2, 2 * r3
            
            D_alpha = abs(C1 * alpha[j] - wolves[i, j])
            D_beta = abs(C2 * beta[j] - wolves[i, j])
            D_delta = abs(C3 * delta[j] - wolves[i, j])
            
            X1 = alpha[j] - A1 * D_alpha
            X2 = beta[j] - A2 * D_beta
            X3 = delta[j] - A3 * D_delta
            
            wolves[i, j] = (X1 + X2 + X3) / 3
            wolves[i, j] = 1 if wolves[i, j] > 0.5 else 0  # Convert to binary
# Evaluate the best solution
best_wolf = alpha
selected_features = [i for i in range(len(best_wolf)) if best_wolf[i] > 0.5]

if len(selected_features) == 0:
    print(" No features selected by BGWO. Using all features as fallback.")
    selected_features = list(range(X.shape[1]))  # fallback: use all features

# Train the final model with GaussianNB using the selected features
final_model = GaussianNB()
final_model.fit(X_train.iloc[:, selected_features], y_train)
y_pred = final_model.predict(X_test.iloc[:, selected_features])

# Print performance metrics
print(f"Selected Features: {selected_features}")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


 No features selected by BGWO. Using all features as fallback.
Selected Features: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 20